In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from hello_agents_llm import HelloAgentsLLM
my_llm = HelloAgentsLLM()

messages = [
    {'role': 'user', 'content': 'who are you'}
]
my_llm.generate(messages=messages)

----------llm invoking start----------
messages:
[{'role': 'user', 'content': 'who are you'}]


APIConnectionError: Connection error.

# 系统定义

输入一组数据, 输出对应的提取内容. 输入格式定义是混乱的, 输出必须为 json, 具体为
```json
{
    "name": "<>",
    "address": "<>",
    "email": "<>",
    "question": "<>"
}
```

eg.
```bash
# input
龙琳，宁夏回族自治区璐市城东林街g座 955491，邮箱 nafan@example.com。小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！
```
```bash
# output
{
    "name": "龙琳",
    "address": "宁夏回族自治区璐市城东林街g座 955491",
    "email": "nafan@example.com",
    "question": "小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！"
}
```


# 测试方法搭建

In [ ]:
import json, re
SYSTEM_PROMPT = """
You are an information extraction and structuring engine.

# Task
The user will provide a piece of free-form text (often Chinese, possibly mixed with English/numbers/symbols). The text may contain: a person’s name, an address, an email, and a main issue/request (complaint, question, feedback, etc.). The input format can be messy, punctuation may be irregular, fields can appear in any order, and there may be extra irrelevant content. Your job is to extract the required fields and output them in a fixed JSON object.

# Output Format (must be strictly followed)
You must output **exactly one JSON object** and nothing else—no explanations, no prefixes/suffixes, no code fences, no Markdown, no comments, and no extra characters. The JSON schema is fixed:
{
  "name": "<>",
  "address": "<>",
  "email": "<>",
  "question": "<>"
}

# Extraction Rules
1) name
- Prefer an explicit personal name (common Chinese names or names following cues like “Name/Contact/Mr./Ms.” etc.).
- If multiple candidates exist, choose the one most likely to be the complainant/asker/reporter.
- If not found, return an empty string "".

2) address
- Extract the most complete physical address (province/city/district/street/building/unit/zip, etc.).
- Do NOT include other fields such as email/phone/name inside the address.
- If multiple addresses exist, prefer the more specific/longer/more detailed one.
- If not found, return "".

3) email
- Detect standard email pattern local@domain (allow letters, digits, dots, underscores, hyphens).
- If multiple emails exist, choose the one most likely belonging to the subject (often near “Email/E-mail/Mail/邮箱” or the first one mentioned).
- If not found, return "".

4) question
- Extract the user’s primary issue/request text (often descriptive sentences and may include emotional wording).
- Remove identity fields (name/address/email) and keep the remaining problem-related content; do NOT paraphrase, summarize, or expand.
- If there are multiple complaint/request segments, merge them into a single string in the original order (preserve original punctuation when reasonable).
- If not found, return "".

# Cleaning & Consistency Requirements
- Output must be valid JSON. Escape double quotes inside values properly (e.g., " -> \").
- Do not guess or fabricate any field. If uncertain, use "".
- Trim meaningless leading/trailing whitespace and optional trailing punctuation where appropriate, but do not over-edit or rewrite content.
- Keep numbers/letters/zip codes in the address exactly as they appear.

# Critical Constraints
- Output ONLY the JSON object (plain text). No other content is allowed.
"""

def prompt_test(llm: HelloAgentsLLM = HelloAgentsLLM(), input:str=""):
    # 组装message
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {'role': 'user', "content": input}
    ]
    response = llm.generate(messages=messages)
    # return json.loads(response)
    ## 可实现, 但是不够鲁邦的方法
    # start = response.find('{')
    # end = response.rfind('}') + 1 # python的切片是左闭右开
    # json_str = response[start:end]
    # start = response.find('</think>')
    json_str = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).replace("\n", "")
    return json.loads(json_str)

def test(llm: HelloAgentsLLM = HelloAgentsLLM(), input:str=""):
    # 组装message
    messages = [
        {"role": "system", "content": "Extract name, address, email, and question"},
        {'role': 'user', "content": input}
    ]
    response = llm.generate(messages=messages)
    # return json.loads(response)
    ## 可实现, 但是不够鲁邦的方法
    # start = response.find('{')
    # end = response.rfind('}') + 1 # python的切片是左闭右开
    # json_str = response[start:end]
    # start = response.find('</think>')
    json_str = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).replace("\n", "")
    return json.loads(json_str)

In [4]:
error_count = 0
success_count = 0

for i in range(100):
    input = "龙琳，宁夏回族自治区璐市城东林街g座 955491，邮箱 nafan@example.com。小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！"
    try:
        if test(input = input):
            success_count+=1
        else:
            error_count +=1
    except Exception as e:
        error_count += 1

print(f"\n测试完成:")
print(f"成功: {success_count}")
print(f"失败: {error_count}")

----------llm invoking start----------
messages:
[{'role': 'system', 'content': 'Extract name, address, email, and question'}, {'role': 'user', 'content': '龙琳，宁夏回族自治区璐市城东林街g座 955491，邮箱 nafan@example.com。小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！'}]
output:
<think>
好的，我需要处理用户提供的这个查询。首先，用户发来了一个关于龙琳的个人信息，包括地址、电话、邮箱，以及小区垃圾和噪音的问题。用户可能是在寻求帮助，或者想表达对某个问题的不满。

首先，我要确认用户的需求是什么。看起来用户可能希望得到关于龙琳的联系方式，或者可能是在询问如何解决小区的问题。但根据提供的信息，用户已经提供了所有必要的信息，包括名字、地址、电话、邮箱，以及问题描述。这可能意味着用户希望将这些信息整理成一个结构化的格式，或者可能是在测试我的能力，确保我能正确提取关键信息。

接下来，我需要检查是否有遗漏的信息。用户提到龙琳，地址是宁夏回族自治区璐市城东林街g座955491，邮箱是nafan@example.com。问题部分提到小区垃圾堆积、噪音扰人、停车难，无法忍受。看起来用户可能希望将这些信息整合到一个回复中，或者可能是在测试我的提取能力。

然后，我需要考虑用户可能的深层需求。用户可能希望得到帮助，比如联系龙琳解决小区的问题，或者可能是在测试我的信息提取能力，确保正确提取所有必要的信息。因此，我需要确保回复中包含所有提供的信息，并且结构清晰，方便用户理解。

最后，我需要确保回复符合用户的要求，即提取出名字、地址、邮箱和问题。同时，保持语言简洁明了，避免冗长。可能用户希望将这些信息以某种方式呈现，比如列表或分段，但根据用户提供的示例，回复可能只需要包含这些部分，而不需要额外解释。
</think>

**提取信息如下：**  
- **姓名**：龙琳  
- **地址**：宁夏回族自治区璐市城东林街g座  
- **联系电话**：955491  
- **邮箱**：nafan@example.com  

**问题描述**：小区垃圾堆积成山，晚上噪音扰人

KeyboardInterrupt: 

In [79]:
error_count = 0
success_count = 0

for i in range(100):
    input = "龙琳，宁夏回族自治区璐市城东林街g座 955491，邮箱 nafan@example.com。小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！"
    try:
        prompt_test(input = input)
        success_count+=1
    except Exception as e:
        error_count += 1

print(f"\n测试完成:")
print(f"成功: {success_count}")
print(f"失败: {error_count}")

----------llm invoking start----------
messages:
[{'role': 'system', 'content': '\nYou are an information extraction and structuring engine.\n\n# Task\nThe user will provide a piece of free-form text (often Chinese, possibly mixed with English/numbers/symbols). The text may contain: a person’s name, an address, an email, and a main issue/request (complaint, question, feedback, etc.). The input format can be messy, punctuation may be irregular, fields can appear in any order, and there may be extra irrelevant content. Your job is to extract the required fields and output them in a fixed JSON object.\n\n# Output Format (must be strictly followed)\nYou must output **exactly one JSON object** and nothing else—no explanations, no prefixes/suffixes, no code fences, no Markdown, no comments, and no extra characters. The JSON schema is fixed:\n{\n  "name": "<>",\n  "address": "<>",\n  "email": "<>",\n  "question": "<>"\n}\n\n# Extraction Rules\n1) name\n- Prefer an explicit personal name (comm

# 数据集处理

In [11]:
import pandas as pd
from datasets import Dataset
ds = Dataset.from_pandas(
    pd.read_json('fake_sft.json')
)

In [12]:
ds

Dataset({
    features: ['system', 'instruction', 'input', 'output'],
    num_rows: 300
})

In [13]:
ds[2]

{'system': '将文本中的name、address、email、question提取出来，以json格式输出，字段为name、address、email、question，值为文本中提取出来的内容。',
 'instruction': '林桂兰：丰都李路V座 458454，邮箱leiwang@example.com。你们小区垃圾成山，噪音扰民，停车位简直是个笑话，这样的管理真是让人火大！',
 'input': '',
 'output': {'address': '丰都李路V座 458454',
  'email': 'leiwang@example.com',
  'name': '林桂兰',
  'question': '你们小区垃圾成山，噪音扰民，停车位简直是个笑话，这样的管理真是让人火大！'}}

# 提词器

In [14]:
from transformers import AutoTokenizer
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    # use_fast=False
)

In [15]:
# tokenizer 处理效果演示, 单纯看一下模板是什么样的
'''
备份输出
<|im_start|>system\n<!><|im_end|>\n + 
<|im_start|>user\n<!><|im_end|>\n + 
<|im_start|>assistant\n<think>\n\n</think>\n\n
'''
messages = [
    {'role': 'system', 'content': '<!>'},
    {'role': 'user', 'content': '<!>'},
    # {'role': 'assistant', 'content': '<!>'}
]
tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # 这里的作用貌似就是是否提前将 think 字段给顶替掉, 那样 llm 就不生成 think 部分了
)


'<|im_start|>system\n<!><|im_end|>\n<|im_start|>user\n<!><|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [55]:
def process_func(input):
    '''
    生成进行 sft 需要的 input_ids, attention_mask, labels
    '''
    # 初始配置
    MAX_LENGTH = 4096
    input_ids, attention_mask, labels = [], [], []

    # 零件, token_ids
    '''
    {
        'system': '将文本中的name、address、email、question提取出来，以json格式输出，字段为name、address、email、question，值为文本中提取出来的内容。',
        'instruction': '林桂兰：丰都李路V座 458454，邮箱leiwang@example.com。你们小区垃圾成山，噪音扰民，停车位简直是个笑话，这样的管理真是让人火大！',
        'input': '',
        'output': '```json\n{\n    "name": "林桂兰",\n    "address": "丰都李路V座 458454",\n    "email": "leiwang@example.com",\n    "question": "你们小区垃圾成山，噪音扰民，停车位简直是个笑话，这样的管理真是让人火大！"\n}\n```'
    }
    '''
    instruction = tokenizer(
        f""
        f"<|im_start|>system\n{input['system']}<|im_end|>\n"
        f"<|im_start|>user\n{input['instruction'] + input['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n",
        add_special_tokens = False
    )
    
    response = tokenizer(
        f"{input['output']}",
        add_special_tokens = False
    )

    # 组装
    input_ids = instruction['input_ids'] + response['input_ids'] + [tokenizer.eos_token_id]
    attention_mask = instruction['attention_mask'] + response['attention_mask'] + [1]
    labels = [-100] * len(instruction['input_ids']) + response['input_ids'] + [tokenizer.eos_token_id]

    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [56]:
process_func(
    ds[2]
)

{'input_ids': [151644,
  8948,
  198,
  44063,
  108704,
  101047,
  606,
  5373,
  4995,
  5373,
  2332,
  5373,
  7841,
  107439,
  99898,
  3837,
  23031,
  2236,
  68805,
  66017,
  3837,
  44931,
  17714,
  606,
  5373,
  4995,
  5373,
  2332,
  5373,
  7841,
  3837,
  25511,
  17714,
  108704,
  15946,
  107439,
  104355,
  43815,
  1773,
  151645,
  198,
  151644,
  872,
  198,
  99463,
  100877,
  99533,
  5122,
  99638,
  71268,
  100339,
  45995,
  53,
  99579,
  220,
  19,
  20,
  23,
  19,
  20,
  19,
  3837,
  93568,
  273,
  37081,
  524,
  35487,
  905,
  1773,
  101900,
  102039,
  100813,
  12857,
  57811,
  3837,
  110310,
  100332,
  69721,
  3837,
  115805,
  101605,
  104104,
  109959,
  3837,
  101893,
  39352,
  101228,
  103973,
  79599,
  26288,
  6313,
  151645,
  198,
  151644,
  77091,
  198,
  151667,
  271,
  151668,
  271,
  13608,
  4995,
  1210,
  364,
  99638,
  71268,
  100339,
  45995,
  53,
  99579,
  220,
  19,
  20,
  23,
  19,
  20,
  19,
  516,


In [57]:
tokenized_id = ds.map(process_func, remove_columns=ds.column_names)

Map: 100%|██████████| 300/300 [00:00<00:00, 1817.16 examples/s]


In [58]:
tokenized_id

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 300
})

In [59]:
tokenized_id[0]

{'input_ids': [151644,
  8948,
  198,
  44063,
  108704,
  101047,
  606,
  5373,
  4995,
  5373,
  2332,
  5373,
  7841,
  107439,
  99898,
  3837,
  23031,
  2236,
  68805,
  66017,
  3837,
  44931,
  17714,
  606,
  5373,
  4995,
  5373,
  2332,
  5373,
  7841,
  3837,
  25511,
  17714,
  108704,
  15946,
  107439,
  104355,
  43815,
  1773,
  151645,
  198,
  151644,
  872,
  198,
  99465,
  105115,
  5122,
  107226,
  18397,
  103840,
  23836,
  113984,
  22697,
  59074,
  67364,
  99463,
  99913,
  70,
  99579,
  220,
  24,
  20,
  20,
  19,
  24,
  16,
  3837,
  93568,
  308,
  2577,
  276,
  35487,
  905,
  1773,
  102039,
  100813,
  109266,
  12857,
  57811,
  3837,
  104030,
  110310,
  100332,
  17340,
  79766,
  99815,
  3837,
  101531,
  99349,
  17447,
  20929,
  99349,
  3837,
  101605,
  101068,
  110313,
  6313,
  151645,
  198,
  151644,
  77091,
  198,
  151667,
  271,
  151668,
  271,
  13608,
  4995,
  1210,
  364,
  107226,
  18397,
  103840,
  23836,
  113984,
 

In [60]:
tokenizer.decode(tokenized_id[0]['input_ids'])

"<|im_start|>system\n将文本中的name、address、email、question提取出来，以json格式输出，字段为name、address、email、question，值为文本中提取出来的内容。<|im_end|>\n<|im_start|>user\n龙琳：宁夏回族自治区璐市城东林街g座 955491，邮箱 nafan@example.com。小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n{'address': '宁夏回族自治区璐市城东林街g座 955491', 'email': 'nafan@example.com', 'name': '龙琳', 'question': '小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！'}<|im_end|>"

In [61]:
tokenizer.decode(
    [i for i in tokenized_id[0]['labels'] if i != -100]
)

"{'address': '宁夏回族自治区璐市城东林街g座 955491', 'email': 'nafan@example.com', 'name': '龙琳', 'question': '小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！'}<|im_end|>"

# 加载模型

In [62]:
from transformers import AutoModelForCausalLM
# 加载预训练模型
model = AutoModelForCausalLM.from_pretrained(
    model_id
).to('cuda')
model

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 473.79it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [63]:
# 原始模型测试
import torch
prompt = "龙琳   ，宁夏回族自治区璐市城东林街g座 955491，nafan@example.com。小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！太插件了阿萨德看见啊啥的健康仨都会撒娇看到撒谎的、"

messages = [
    {"role": "system", "content": "将文本中的name、address、email、question提取出来, "},
    {"role": "user", "content": prompt}
]

inputs = tokenizer.apply_chat_template(messages,
                                       add_generation_prompt=True,
                                       tokenize=True,
                                       return_tensors="pt",
                                       return_dict=True,
                                       enable_thinking=False).to('cuda')

gen_kwargs = {"max_length": 2500, "do_sample": True, "top_k": 1}
outputs = model.generate(**inputs, **gen_kwargs)
outputs = outputs[:, inputs['input_ids'].shape[1]:]
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

- name: 龙琳  
- address: 宁夏回族自治区璐市城东林街g座  
- email: nafan@example.com  
- question: 小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！太插件了阿萨德看见啊啥的健康仨都会撒娇看到撒谎的


In [64]:
model.enable_input_require_grads()

In [65]:
from peft import LoraConfig, TaskType, get_peft_model
# Lora 配置
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=False, # 训练模式
    r=8, # Lora 秩
    lora_alpha=32, # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1# Dropout 比例
)
config
model = get_peft_model(model, config)

In [66]:
model.print_trainable_parameters()  # 模型参数训练量只有0.8395%

trainable params: 5,046,272 || all params: 756,678,656 || trainable%: 0.6669


In [67]:
from transformers import TrainingArguments
args = TrainingArguments(
    output_dir="Qwen3_instruct_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=1,
    num_train_epochs=3,
    save_steps=50,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
)

In [68]:
import swanlab
from swanlab.integration.transformers import SwanLabCallback

# 实例化SwanLabCallback
swanlab_callback = SwanLabCallback(
    project="Qwen3-Lora",
    experiment_name="Qwen3-0.6B-extarct-lora-2"
)

In [69]:
from transformers import Trainer, DataCollatorForSeq2Seq
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_id,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    # callbacks=[swanlab_callback]
)

In [70]:
trainer.train()

Step,Training Loss
1,0.411653
2,0.227204
3,0.103170
4,0.042773
5,0.010349
6,0.025856
7,0.007546
8,0.011193
9,0.006716
10,0.024940


TrainOutput(global_step=21, training_loss=0.04578522856657704, metrics={'train_runtime': 107.5556, 'train_samples_per_second': 8.368, 'train_steps_per_second': 0.195, 'total_flos': 566174907629568.0, 'train_loss': 0.04578522856657704, 'epoch': 3.0})

In [ ]:
import torch

# model = AutoModelForCausalLM.from_pretrained(model_id).to('cuda')

prompt = "龙琳   ，宁夏回族自治区璐市城东林街g座 955491，nafan@example.com。小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！太插件了阿萨德看见啊啥的健康仨都会撒娇看到撒谎的、"

messages = [
    {"role": "system", "content": "将文本中的name、address、email、question提取出来, "},
    {"role": "user", "content": prompt}
]

inputs = tokenizer.apply_chat_template(messages,
                                       add_generation_prompt=True,
                                       tokenize=True,
                                       return_tensors="pt",
                                       return_dict=True,
                                       enable_thinking=False).to('cuda')

gen_kwargs = {"max_length": 250, "do_sample": True, "top_k": 1}
with torch.no_grad():
    outputs = model.generate(**inputs, **gen_kwargs)
    outputs = outputs[:, inputs['input_ids'].shape[1]:]
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 464.73it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


- name: 龙琳  
- address: 宁夏回族自治区璐市城东林街g座  
- email: nafan@example.com  
- question: 小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！太插件了阿萨德看见啊啥的健康仨都会撒娇看到撒谎的


In [36]:
messages = [{'role': 'user', 'content': 'who are you'}]

format_messages = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
)

input_ids = tokenizer(
    format_messages,
    return_tensors = 'pt'
).to('cuda')

input_ids

{'input_ids': tensor([[151644,    872,    198,  14623,    525,    498, 151645,    198, 151644,
          77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [103]:
output_ids = model.generate(
    **input_ids,
    max_new_tokens = 512
)
output_ids

tensor([[151644,    872,    198,  14623,    525,    498, 151645,    198, 151644,
          77091,    198, 151667,  38297,  21806,  87318,  33487,   7662,   7662,
          38297,   1390,   6711,  21806,  14582,  38297,   3988,  83291,  12478,
          10294,  14582,  14582,  15846,  69769,  38297,   7662,  21806,  14582,
          14582,  12478,  21806,      9,      9,  14582,    353,  69769,  38297,
          38297,  61832,    477,  61832,  61832,  61832,   3988,  38297,  38297,
          14582,  38297,  69769,  14582,  12478,  12478,    353,  38297,  14582,
          15846,  21806,  15846,  72390,  38297,  12478,    353,    353,    353,
            353,    353,    262,  15846,  38297,  15846,  14582,  21806,  38297,
          38297,   3988,  38297,  38297,  61832,   3988,  12478,  38297,   1390,
           1479,  14096,  38297,  38297,  38297,  69769,  33487,  38297,  38297,
           6711,  10294,  21806,  38297,  38297,  38297,  38297,   7662,  14582,
           3988,  21806,  38

In [104]:
# 解析模型输出: 先给你看看初始内容
tokenizer.batch_decode(
    output_ids,
)

['<|im_start|>user\nwho are you<|im_end|>\n<|im_start|>assistant\n<think> Instructions Answer SHIPPINGComparison Description Description Instructionsarentlying AnswerQuestion Instructions Name글 Solution ReviewQuestionQuestion QuestionExplanation Instructions Description AnswerQuestionQuestion Solution Answer**Question *Explanation Instructions Instructionslicantsestlicantslicantslicants Name Instructions InstructionsQuestion InstructionsExplanationQuestion Solution Solution * InstructionsQuestion Question Answer QuestionPublication Instructions Solution * * * * *    Question Instructions QuestionQuestion Answer Instructions Instructions Name Instructions Instructionslicants Name Solution InstructionsarentEDStudent Instructions Instructions InstructionsExplanationComparison Instructions Instructionslying Review Answer Instructions Instructions Instructions Instructions DescriptionQuestion Name Answer InstructionsQuestion Answer글글기비자 Instructions자자 Answer Instructions Instructionslicants